# M3 Event Classifier (`m3_event_clf`) — DeBERTa-v3-base 4-Class + ONNX `fp32`

- **Task (`AR-08`):** 4-class classification (`actual_event`, `hypothetical`, `negated`, `not_relevant`) on candidate severe-finding sentences (`going_concern`, `default_or_recall`, `regulator_order`, `fraud`, `qualified_audit`) with surrounding $\pm 1$ sentence context.
- **Expected Compute Units:**
  - **`DEBUG = True` (T4 smoke run):** ~3 min (~0.08 Colab units). May run on unverified `ai` labels in `DEBUG=True` mode only to validate pipeline, checkpointing, ONNX export, and `metrics.json` schema.
  - **`DEBUG = False` (G4 full 3-seed run, `SEEDS = [13, 42, 2026]`, 4 epochs/seed):** ~20 min (**2.90 G4 units** at 8.71 units/hr). Requires Akshat's verified L2 labels (`human_verified_ai`).
- **Outputs on Drive (`/content/drive/MyDrive/finsight_p4/m3_event_clf/`):**
  - Epoch checkpoints: `epoch-<k>/` and `seed-<s>/epoch-<k>/`
  - Exported ONNX bundle: `onnx_fp32/{model.onnx, tokenizer.json, tokenizer_config.json, threshold.json, SHA256SUMS, WEIGHTS.md}`
  - Contract metrics: `metrics.json` (copy back to `eval_results/p4/argon/m3_event_clf/metrics.json`)

In [ ]:
# First cell of every ARGON notebook (from ml/handoff/colab/setup_cell.py)
import json
import os
import pathlib
import random
import shutil
import subprocess
import sys

HF_REPO = "AkshatTm/finsight-gpu-jobs"  # private dataset repo (C-ADR-13)
HF_PREFIX = "phase4/handoff"  # folder in that repo; revision pinned in ml/handoff/README.md
HF_REVISION = "a285902d91a5557cf6d6d7cf23a9e266410d6c3d"  # upload_large revision in MANIFEST.json
HF_PACK_REVISION = "af028c61cde3f0aee6e52fc40593ac47b2a7aa1b"  # upload_pack commit in export_stats.json (contains MANIFEST.json + full pack)
JOB = "m3_event_clf"
DEBUG = True  # True = tiny slice for T4 debug run; set False on G4 ONLY after DEBUG passes
SEEDS = [13, 42, 2026]
RESUME = True  # continue from the newest checkpoint on Drive if one exists

ROOT = pathlib.Path("/content")
PACK = ROOT / "handoff"
DRIVE = pathlib.Path("/content/drive/MyDrive/finsight_p4")
CKPT_DIR = DRIVE / JOB


def mount_drive() -> None:
    try:
        from google.colab import drive  # type: ignore

        drive.mount("/content/drive")
    except Exception as exc:
        print("Drive not mounted:", exc)
    CKPT_DIR.mkdir(parents=True, exist_ok=True)


def hf_token() -> str:
    try:
        from google.colab import userdata  # type: ignore

        tok = userdata.get("HF_TOKEN")
        if tok:
            return tok.strip()
    except Exception:
        pass
    if "HF_TOKEN" in os.environ and os.environ["HF_TOKEN"].strip():
        return os.environ["HF_TOKEN"].strip()
    raise RuntimeError(
        "HF_TOKEN not found! In Colab, click the key icon (Secrets) on the left sidebar, "
        "add HF_TOKEN, and toggle 'Notebook access' ON."
    )


def fetch_pack() -> pathlib.Path:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)
    from huggingface_hub import snapshot_download

    token = hf_token()
    hf_cache_dir = ROOT / "_hf"
    # First pull the full pack commit (af028c6...) which includes MANIFEST.json, colab/, eval/, data/
    for rev in (HF_PACK_REVISION, None, HF_REVISION):
        try:
            snapshot_download(
                repo_id=HF_REPO,
                repo_type="dataset",
                revision=rev,
                token=token,
                allow_patterns=[f"{HF_PREFIX}/**", f"{HF_PREFIX}/*"],
                local_dir=str(hf_cache_dir),
            )
            if (hf_cache_dir / HF_PREFIX / "data" / "MANIFEST.json").exists():
                break
        except Exception as exc:
            print(f"snapshot_download (rev={rev}) note:", exc)

    # Ensure the 3 large files from HF_REVISION are also present
    for big_file in ("classifier_train.jsonl.gz", "risk_bank.parquet", "student_v1_train.jsonl.gz"):
        if not (hf_cache_dir / HF_PREFIX / "data" / big_file).exists():
            snapshot_download(
                repo_id=HF_REPO,
                repo_type="dataset",
                revision=HF_REVISION,
                token=token,
                allow_patterns=[f"{HF_PREFIX}/data/{big_file}"],
                local_dir=str(hf_cache_dir),
            )

    src = hf_cache_dir / HF_PREFIX
    assert (src / "data" / "MANIFEST.json").exists(), f"MANIFEST.json not found in {src}/data!"
    if PACK.exists():
        shutil.rmtree(PACK)
    shutil.copytree(src, PACK)
    return PACK


def verify_manifest() -> None:
    import hashlib

    manifest = json.loads((PACK / "data" / "MANIFEST.json").read_text(encoding="utf-8"))
    for f in manifest["files"]:
        h = hashlib.sha256((PACK / "data" / f["path"]).read_bytes()).hexdigest()
        assert h == f["sha256"], f"checksum mismatch: {f['path']}"


def install_deps() -> None:
    req_file = PACK / "colab" / "requirements_colab.txt"
    res = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(req_file)],
        capture_output=True,
        text=True,
    )
    if res.returncode != 0:
        print("pip install -r requirements_colab.txt stderr:\n", res.stderr)
        res.check_returncode()


def set_seed(seed: int) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import numpy as np

        np.random.seed(seed)
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


def latest_checkpoint() -> pathlib.Path | None:
    ckpts = sorted(CKPT_DIR.glob("epoch-*"), key=lambda p: int(p.name.split("-")[1]))
    return ckpts[-1] if RESUME and ckpts else None


mount_drive()
fetch_pack()
verify_manifest()
install_deps()
sys.path.insert(0, str(PACK / "eval" / "vendor"))
print("pack ready:", PACK, "| DEBUG =", DEBUG, "| resume from:", latest_checkpoint())

In [ ]:
# Cell 2: Load L2 event_labels dataset (or build directly from PACK/data/serious_finding_hits.jsonl.gz on Colab), apply Akshat's verification CSV if present, and enforce 0 leakage
import csv
import gzip
import re
from collections import Counter, defaultdict

LABELS = ["actual_event", "hypothetical", "negated", "not_relevant"]
LABEL2ID = {lbl: idx for idx, lbl in enumerate(LABELS)}
ID2LABEL = dict(enumerate(LABELS))
MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LENGTH = 256

FIRED_ACTUAL_EVENTS: set[str] = {
    # Train / Dev fired actual_event:
    "hit000111",
    "hit000112",
    "hit000114",
    "hit000115",
    "hit000116",
    "hit000118",
    "hit000119",
    "hit000120",
    "hit000122",
    "hit000123",
    "hit000124",
    "hit000126",
    "hit000127",
    "hit000128",
    "hit000130",
    "hit000131",
    "hit000132",
    "hit000134",
    "hit000135",
    "hit000137",
    "hit000139",
    "hit000140",
    "hit000142",
    "hit000143",
    "hit000144",
    "hit000146",
    "hit000147",  # afcons-infrastructure-2024: Joint Operations / Subsidiaries going concern material uncertainty
    "hit000161",  # akme-fintrade-india-2024: delays in repayment of principal/interest across lenders
    "hit000260",
    "hit000263",
    "hit000267",
    "hit000268",
    "hit000274",
    "hit000275",
    "hit000280",
    "hit000284",
    "hit000288",
    "hit000292",
    "hit000294",  # apeejay-surrendra-park-hotels-2024: loan repayment delays/defaults + going concern uncertainty
    "hit000346",  # ather-energy-2025: SEBI final order in respect of insider trading violations
    "hit000380",  # baazar-style-retail-2024: adjudicating officer imposed monetary penalty of Rs 1.30M
    "hit000391",
    "hit000392",  # bajaj-housing-finance-2024: NSE/BSE/SEBI imposed penalties on group entities
    "hit000457",
    "hit000458",  # bls-e-services-2024: Independent Director associated with company debarred by SEBI
    "hit000462",  # bls-e-services-2024: Starfin failed to pay rent and defaulted under sub-lease
    "hit000465",
    "hit000467",
    "hit000468",
    "hit000470",  # bluestone-jewellery-and-lifestyle-2025: delays in repayment of loans during FY24 and FY23
    "hit000489",  # brainbees-solutions-2024: statutory auditors of Subsidiaries observed going concern uncertainties
    "hit000548",  # capital-small-finance-bank-2024: RBI imposed fine of Rs 0.05M on our Bank
    "hit000560",  # ceigall-india-2024: Director Arun Goyal was debarred from accessing securities market in past
    "hit000608",  # crizac-2025: Promoter was director of companies struck off by RoC
    "hit000682",  # dev-accelerator-2025: Auditor report emphasis of matter and CARO modifications
    "hit000729",  # emcure-pharmaceuticals-2024: Subsidiary Gennova had delays in loan repayment in FY22
    "hit000802",  # exicom-tele-systems-2024: Examination report emphasis of matter and CARO adverse remarks
    "hit000878",
    "hit000879",
    "hit000880",
    "hit000885",
    "hit000894",
    "hit000895",  # gem-aromatics-2025: Promoter Group entity GDPPL and individuals categorised as wilful defaulter by SBI
    "hit000909",  # gk-energy-2025: Statutory Auditor included observations under CARO 2020
    "hit001224",
    "hit001225",
    "hit001226",  # jain-resource-recycling-2025: SEBI orders imposing monetary penalties on Kamlesh Jain
    "hit001237",
    "hit001238",
    "hit001239",
    "hit001242",
    "hit001243",
    "hit001246",
    "hit001247",
    "hit001251",
    "hit001254",
    "hit001258",  # jana-small-finance-bank-2024: Qualified opinion & going concern material uncertainty
    "hit001264",
    "hit001266",  # jaro-institute-2025: loan servicing delays + SEBI adjudication order imposing Rs 1M penalty on Promoter
    "hit001293",
    "hit001295",
    "hit001296",
    "hit001297",  # jsw-cement-2025: Subsidiary going concern uncertainty + ED/SEBI penalty orders
    "hit001309",
    "hit001310",
    "hit001311",
    "hit001312",
    "hit001320",  # juniper-hotels-2024: Subsidiary CHPL witnessed delays in loan repayment & strategic debt restructuring
    "hit001424",  # laxmi-india-finance-2025: Company debarred from BSE Electronic Book Platform up to July 2024
    "hit001579",
    "hit001580",
    "hit001585",
    "hit001589",
    "hit001594",
    "hit001599",
    "hit001601",  # medi-assist-healthcare-services-2024: Disclaimer of opinion in FY21 audit report
    "hit001671",
    "hit001675",
    "hit001676",
    "hit001677",
    "hit001679",
    "hit001680",  # national-securities-depository-2025: SEBI disgorgement/penalty/adverse orders against NSDL
    "hit001743",  # ntpc-green-energy-2024: CARO observation on loan allocation
    "hit001794",
    "hit001795",
    "hit001800",
    "hit001802",  # one-mobikwik-systems-2024: RBI penalty of Rs 10M + Stop Operations Order + fastag fraud
    "hit001929",
    "hit001931",
    "hit001932",  # platinum-industries-2024: Promoter group members defaulted on Indian Overseas Bank loans
    "hit001993",
    "hit001995",
    "hit001998",
    "hit002001",
    "hit002002",  # quadrant-future-tek-2024: Qualified audit opinion for FY22 (inventory / Ind AS 2)
    "hit002070",
    "hit002071",  # rubicon-research-2025: USFDA Form 483 observations / regulatory action
    "hit002091",  # sagility-india-2024: SEBI order dated July 24, 2024
    "hit002133",  # sanstar-2024: Subsidiary regulatory/compounding action
    "hit002143",
    "hit002144",  # saraswati-saree-depot-2024: Audit report for FY22-FY24 includes Qualified Opinion
    "hit002183",
    "hit002184",
    "hit002185",
    "hit002187",
    "hit002189",
    "hit002190",
    "hit002191",  # senores-pharmaceuticals-2024: Erstwhile statutory auditors issued qualified opinion for FY22
    "hit002283",
    "hit002284",
    "hit002285",
    "hit002287",
    "hit002290",
    "hit002291",
    "hit002295",
    "hit002296",
    "hit002298",
    "hit002299",  # smartworks-coworking-spaces-2025: Disclaimed/qualified opinion on IFC + Company/Promoter loan defaults (NPA)
    "hit002310",  # solarworld-energy-solutions-2025: SEBI adjudication order dated Dec 31, 2019
    "hit002382",
    "hit002383",
    "hit002384",  # stanley-lifestyles-2024: Delays in repayment of vehicle loans by Company and Subsidiary
    "hit002429",
    "hit002431",
    "hit002433",  # suraksha-diagnostic-2024: Basis for Qualified Opinion in Audit Report (FY23)
    "hit002495",
    "hit002496",
    "hit002514",
    "hit002516",  # tbo-tek-2024: ED Show Cause Notice for FEMA contraventions; RBI compounding filed
    "hit002538",
    "hit002539",
    "hit002540",
    "hit002541",
    "hit002546",
    "hit002547",
    "hit002552",
    "hit002553",  # tolins-tyres-2024: Promoter Group member Jose Thomas / Coromandel Leathers on CIBIL wilful defaulter list + loan delays
    "hit002689",  # vms-tmt-2025: Group entity Eternal Motors Pvt Ltd defaulted on loan repayment
    "hit002692",
    "hit002693",
    "hit002694",
    "hit002696",
    "hit002697",
    "hit002698",
    "hit002699",
    "hit002700",
    "hit002703",
    "hit002706",
    "hit002707",
    "hit002708",
    "hit002709",
    "hit002712",
    "hit002713",
    "hit002716",
    "hit002717",
    "hit002718",
    "hit002721",
    "hit002722",  # vodafone-idea-2024: Material uncertainty related to going concern in audit reports + employee fraud FIR
    # Test-slice fired actual_event:
    "hit000203",
    "hit000204",
    "hit000205",  # alpine-texworld-2026: SEBI adjudicating officer imposed monetary penalties on Promoter Group member
    "hit000214",  # amagi-media-labs-2026: Statutory auditors adverse remark on IFC / audit trail
    "hit000238",  # aequs-2025: Delays in repayment of borrowings by Subsidiaries in the past
    "hit000537",  # capillary-technologies-india-2025: Delays in repayment of borrowing in FY23
    "hit000612",
    "hit000613",
    "hit000616",
    "hit000617",
    "hit000618",
    "hit000619",
    "hit000620",
    "hit000621",
    "hit000622",
    "hit000626",
    "hit000627",
    "hit000628",
    "hit000629",  # csm-technologies-2026: Subsidiary audit reports substantial doubt on going concern
    "hit000918",
    "hit000919",
    "hit000921",
    "hit000922",  # glass-wall-systems-india-2026: Qualified opinion in audit reports
    "hit001057",
    "hit001059",
    "hit001060",
    "hit001061",
    "hit001062",
    "hit001063",  # hdb-financial-services-2025: ED, RBI, and SEBI orders imposing penalties on Bank / Company
    "hit001079",  # hexagon-nutrition-2026: Subsidiary audit reports material uncertainty on going concern
    "hit001122",
    "hit001123",  # inox-clean-energy-2026: Delays in repayment of loans / adverse CARO remarks
    "hit001555",
    "hit001558",  # manipal-health-enterprises-2026: Qualified opinion on financial statements
    "hit001614",  # meesho-2025: Adverse remark on backup / audit trail in auditor's report
    "hit001625",
    "hit001627",
    "hit001629",
    "hit001630",
    "hit001631",
    "hit001632",
    "hit001634",
    "hit001635",  # molbio-diagnostics-2026: Disclaimer/Qualified audit opinion + Company defaulted in repayment of dues to lenders
    "hit001687",
    "hit001688",
    "hit001689",
    "hit001690",
    "hit001691",
    "hit001692",
    "hit001693",
    "hit001694",
    "hit001695",
    "hit001696",
    "hit001698",  # national-stock-exchange-of-india-2026: SEBI penalty orders (Colocation, Dark Fibre, Governance)
    "hit001703",
    "hit001704",  # nephrocare-health-services-2025: Procurement fraud via spoofed email
    "hit001706",
    "hit001711",
    "hit001712",  # nityas-gems-and-jewellery-india-2026: FIR filed for misappropriation of inventory
    "hit001785",  # onemi-technology-solutions-2026: RBI imposed monetary penalty on subsidiary
    "hit001892",
    "hit001893",
    "hit001894",
    "hit001896",
    "hit001897",
    "hit001900",  # physicswallah-2025: Subsidiary iNeuron going concern uncertainty + CARO/IFC remarks
    "hit001904",
    "hit001905",
    "hit001907",
    "hit001909",
    "hit001911",
    "hit001914",
    "hit001919",
    "hit001923",  # pine-labs-2025: Adverse opinion on IFC + delays in loan repayment in FY23
    "hit002042",
    "hit002043",  # rays-of-belief-2026: FIR against ex-employee for misappropriation of proprietary data
    "hit002152",  # sbi-funds-management-2026: SEBI order imposing Rs 250M monetary penalty on Group Company
    "hit002216",  # shah-investor-s-home-2026: SEBI imposed penalty of Rs 4,00,000 on Company
    "hit002267",
    "hit002275",
    "hit002276",  # skyways-air-services-2026: Delays in loan repayment + fraud involving misappropriation of funds by senior employees
    "hit002341",
    "hit002342",
    "hit002348",
    "hit002349",  # ss-retail-2026: SEBI Adjudication Order imposing Rs 2.50M penalty
    "hit002755",
    "hit002759",
    "hit002760",
    "hit002762",  # waterways-leisure-tourism-2026: Material uncertainty related to going concern (FY24)
}

# B. `blocked_soft` / `blocked_no_debt` rows that are genuine `actual_event` rescued by audit:
BLOCKED_ACTUAL_EVENTS: set[str] = {
    # Train / Dev rescued actual_event (65 rows):
    "hit000073",
    "hit000076",
    "hit000078",  # afcons-infrastructure-2024: Joint Operations audit reports contain going concern uncertainty
    "hit000168",  # akme-fintrade-india-2024: NHB imposed penalty of Rs 1,29,800 for non-compliance
    "hit000276",
    "hit000277",  # apeejay-surrendra-park-hotels-2024: Audit report drew attention to going concern uncertainty
    "hit000278",  # apeejay-surrendra-park-hotels-2024: Delays in repayment of credit facilities in FY21-23
    "hit000389",
    "hit000390",  # bajaj-housing-finance-2024: BSE and RBI imposed fine/penalty on the Company
    "hit000469",  # bluestone-jewellery-and-lifestyle-2025: Overdue in repayment of loans (NCDs) in FY24/FY23
    "hit000490",
    "hit000491",  # brainbees-solutions-2024: Subsidiary auditors observed material uncertainty on going concern
    "hit000561",  # ceigall-india-2024: Past book running lead manager debarred by SEBI order
    "hit000643",  # dee-development-engineers-2024: SEBI adjudication order imposed penalty of Rs 6,00,000
    "hit000703",  # dr-agarwal-s-healthcare-2025: BSE/NSE imposed fines on listed subsidiary for LODR non-compliance
    "hit000765",
    "hit000766",  # epack-durable-2024: Qualified audit opinion on internal financial controls
    "hit000800",  # exicom-tele-systems-2024: Regulatory penalty order against Company
    "hit000801",  # exicom-tele-systems-2024: Subsidiary audit report going concern material uncertainty
    "hit000886",
    "hit000887",  # gem-aromatics-2025: CARO ix(a) reported delays in loan repayment in FY23 and FY24
    "hit001228",
    "hit001229",  # jain-resource-recycling-2025: SEBI adjudication order imposing penalty
    "hit001235",  # jana-small-finance-bank-2024: Audit report included going concern material uncertainty
    "hit001236",
    "hit001241",
    "hit001245",
    "hit001249",
    "hit001250",
    "hit001253",  # jana-small-finance-bank-2024: Qualified opinion paragraphs in auditor's reports
    "hit001334",  # jyoti-cnc-automation-2024: Subsidiary Huron Graffenstaden SAS going concern uncertainty
    "hit001581",
    "hit001582",
    "hit001583",
    "hit001584",
    "hit001586",
    "hit001592",
    "hit001595",
    "hit001596",
    "hit001597",
    "hit001598",
    "hit001600",
    "hit001602",  # medi-assist-healthcare-services-2024: Disclaimer / qualified opinion on FY21 statements
    "hit001841",  # orkla-india-2025: Subsidiary Rasoi Magic Foods CARO ix(a) default/delay in FY23
    "hit001996",
    "hit001997",
    "hit001999",
    "hit002003",  # quadrant-future-tek-2024: Basis for Qualified Opinion in FY22 audit report
    "hit002186",  # senores-pharmaceuticals-2024: Basis for Qualified Opinion in FY22 audit report
    "hit002286",
    "hit002300",
    "hit002301",
    "hit002302",
    "hit002303",  # smartworks-coworking-spaces-2025: Qualified opinion in statutory auditor's report
    "hit002428",
    "hit002430",
    "hit002432",  # suraksha-diagnostic-2024: Qualified opinion in audit report
    "hit002695",
    "hit002701",
    "hit002702",
    "hit002704",
    "hit002705",  # vodafone-idea-2024: Material uncertainty exists casting doubt on going concern
    "hit002774",
    "hit002776",  # wework-india-management-2025: Qualified opinion / material weakness in IFC
    # Test-slice rescued actual_event (32 rows):
    "hit000369",  # aye-finance-2026: Instances of delay in payment of borrowings
    "hit000539",
    "hit000542",  # capillary-technologies-india-2025: CARO ix(a) reported delays in loan repayment
    "hit000920",  # glass-wall-systems-india-2026: Basis for Qualified Opinion in auditor's report
    "hit001092",
    "hit001093",  # hexagon-nutrition-2026: Subsidiary audit report going concern material uncertainty
    "hit001556",
    "hit001559",  # manipal-health-enterprises-2026: Basis for Qualified Opinion in auditor's report
    "hit001626",
    "hit001628",
    "hit001633",  # molbio-diagnostics-2026: Basis for Qualified Opinion in auditor's report
    "hit001710",  # nityas-gems-and-jewellery-india-2026: FIR filed in relation to misappropriation of inventory
    "hit001895",
    "hit001898",
    "hit001899",  # physicswallah-2025: Qualified opinion / going concern uncertainty of subsidiary
    "hit001906",
    "hit001908",
    "hit001912",
    "hit001913",
    "hit001917",
    "hit001918",
    "hit001920",
    "hit001922",
    "hit001924",  # pine-labs-2025: Delay in loan repayment in FY23 + Qualified opinion on IFC
    "hit002173",
    "hit002174",  # sedemac-mechatronics-2026: Regulatory penalty / compounding order
    "hit002343",
    "hit002350",  # ss-retail-2026: Regulatory / statutory penalty order
    "hit002756",
    "hit002757",
    "hit002758",  # waterways-leisure-tourism-2026: Material uncertainty related to going concern for FY24
}

# C. Additional genuine `actual_event` rows mined from Risk Factors of non-excluded train/dev RHPs
#    (where `serious_finding_hits` regexes did not trigger or the IPO was pre-2024 corpus):
MINED_TRAIN_DEV_ACTUAL_EVENTS: list[dict] = [
    {
        "id": "rf_ae_0001",
        "ipo_id": "swiggy-2024",
        "slice": "train",
        "page": 56,
        "check_id": "fraud",
        "sentence": "In another instance, the Company, during Fiscal 2024, identified embezzlement of funds in our Material Subsidiary, Scootsy, by a former employee amounting to ₹326.77 million during the period August 2021 to February 2022.",
        "context_before": "Our business operations are susceptible to risks relating to fraud, misappropriation and employee misconduct.",
        "context_after": "We have initiated legal proceedings and taken disciplinary action in connection with the same.",
        "rules_decision": "missed_by_regex",
    },
    {
        "id": "rf_ae_0002",
        "ipo_id": "baazar-style-retail-2024",
        "slice": "train",
        "page": 65,
        "check_id": "regulator_order",
        "sentence": "The RBI has issued a show cause notice dated February 17, 2023 (“SCN”), to Boast Traders Private Limited (formerly known as Acme Private Limited) (“Boast Traders”).",
        "context_before": "Certain regulatory proceedings have been initiated in relation to entities merged with or connected to our Company.",
        "context_after": "Any adverse outcome in relation to such show cause notice may adversely affect our reputation and business.",
        "rules_decision": "missed_by_regex",
    },
    {
        "id": "rf_ae_0003",
        "ipo_id": "canara-hsbc-life-insurance-company-2025",
        "slice": "dev",
        "page": 49,
        "check_id": "regulator_order",
        "sentence": "In the past, pursuant to an onsite inspection of our Company, carried out between July 31, 2012 and August 9, 2012, the IRDAI issued an inspection report dated November 5, 2012 and thereafter a show cause notice on February 14, 2014.",
        "context_before": "We are subject to periodic inspections and regulatory supervision by the IRDAI in the ordinary course of our insurance business.",
        "context_after": "Subsequent to our submissions, IRDAI issued final directions and levied regulatory charges in respect of certain observations.",
        "rules_decision": "missed_by_regex",
    },
    {
        "id": "rf_ae_0004",
        "ipo_id": "go-digit-general-insurance-2024",
        "slice": "train",
        "page": 58,
        "check_id": "regulator_order",
        "sentence": "IRDAI has passed an order dated May 2, 2024, wherein IRDAI has imposed a penalty of ₹ 10.00 million on our Company.",
        "context_before": "IRDAI, through show cause notice dated August 24, 2021 observed that addition of the Optional Cover to the Product falls within the purview of life insurance business.",
        "context_after": "Our Company has complied with the directions issued under the said IRDAI order.",
        "rules_decision": "missed_by_regex",
    },
    {
        "id": "rf_ae_0005",
        "ipo_id": "go-digit-insurance-2024",
        "slice": "train",
        "page": 58,
        "check_id": "regulator_order",
        "sentence": "IRDAI, through show cause notice dated August 24, 2021 observed that addition of the Optional Cover to the Product falls a within the purview of ‘life insurance business’ under Section 2(11) of the Insurance Act, as it offers ‘death benefit’ which cannot be offered by our Company, advised immediate withdrawal.",
        "context_before": "We are subject to extensive regulatory oversight by the IRDAI regarding our insurance product filings.",
        "context_after": "IRDAI has passed an order dated May 2, 2024, wherein IRDAI has imposed a penalty of ₹ 10.00 million on our Company.",
        "rules_decision": "missed_by_regex",
    },
    {
        "id": "rf_ae_0006",
        "ipo_id": "medi-assist-healthcare-services-2024",
        "slice": "train",
        "page": 42,
        "check_id": "regulator_order",
        "sentence": "Based on the responses to the observations issued by the IRDAI after its inspection of one of our Subsidiaries, Medi Assist TPA, the IRDAI issued a letter of advisory dated June 4, 2020 and a show cause notice dated June 4, 2020 to Medi Assist TPA, seeking clarifications on certain alleged violations.",
        "context_before": "Our TPA Subsidiaries are subject to periodic inspections by the IRDAI.",
        "context_after": "In addition, based on the responses to the inspection report dated May 15, 2019 issued by IRDAI to one of our Subsidiaries, Raksha TPA, the IRDAI issued a show cause notice dated July 10, 2020 to Raksha TPA.",
        "rules_decision": "missed_by_regex",
    },
    {
        "id": "rf_ae_0007",
        "ipo_id": "medi-assist-healthcare-services-2024",
        "slice": "train",
        "page": 42,
        "check_id": "regulator_order",
        "sentence": "In addition, based on the responses to the inspection report dated May 15, 2019 issued by IRDAI to one of our Subsidiaries, Raksha TPA, the IRDAI issued a show cause notice dated July 10, 2020 to Raksha TPA.",
        "context_before": "Based on the responses to the observations issued by the IRDAI after its inspection of one of our Subsidiaries, Medi Assist TPA, the IRDAI issued a letter of advisory dated June 4, 2020 and a show cause notice dated June 4, 2020 to Medi Assist TPA.",
        "context_after": "IRDAI subsequently issued warnings and directions to ensure strict compliance with TPA regulations.",
        "rules_decision": "missed_by_regex",
    },
    {
        "id": "rf_ae_0008",
        "ipo_id": "midwest-2025",
        "slice": "dev",
        "page": 42,
        "check_id": "going_concern",
        "sentence": "Consequent to the admission into CIRP, BEML Midwest Limited ceased to be a subsidiary of our Company with effect from September 26, 2022 • The subsidiary auditors also qualified matters related to going concern in their respective audit reports.",
        "context_before": "Our joint venture / subsidiary BEML Midwest Limited was admitted into Corporate Insolvency Resolution Process under the IBC.",
        "context_after": "Refer Note no 42 (A) to the Consolidated Ind AS financial statements for further details.",
        "rules_decision": "missed_by_regex",
    },
    {
        "id": "rf_ae_0009",
        "ipo_id": "national-securities-depository-2025",
        "slice": "dev",
        "page": 54,
        "check_id": "regulator_order",
        "sentence": "August 2, 2023 SEBI issued a warning letter to our Company for non-compliance with SEBI circular no. SEBI/HO/MRD2/DCAP/CIR/P/2021/622.",
        "context_before": "Our Company is inspected annually by SEBI and has received regulatory communications and administrative warnings in the past.",
        "context_after": "October 30, 2023 SEBI issued a warning letter to our Company for non-compliance with SEBI circular.",
        "rules_decision": "missed_by_regex",
    },
    {
        "id": "rf_ae_0010",
        "ipo_id": "national-securities-depository-2025",
        "slice": "dev",
        "page": 55,
        "check_id": "regulator_order",
        "sentence": "December 14, 2023 Pursuant to the onsite inspection of our Company during FY 2022-23, SEBI issued a warning letter for non-compliances noted in relation to the constitution/functioning of the management committee.",
        "context_before": "October 30, 2023 SEBI issued a warning letter to our Company for non-compliance with SEBI circular.",
        "context_after": "Our Company placed the warning letter before the Board and took corrective actions reported to SEBI.",
        "rules_decision": "missed_by_regex",
    },
    {
        "id": "rf_ae_0011",
        "ipo_id": "sanathan-textiles-2024",
        "slice": "train",
        "page": 58,
        "check_id": "regulator_order",
        "sentence": "SEBI, on January 31, 2022, issued a show cause notice, bearing reference number NRO/AK/ASR/3878/1/2022 in terms of Rule 4 (1) of SEBI (Procedure for Holding Inquiry and Imposing Penalties by Adjudicating Officer) Rules, 1995 to Dineshkumar Vrajdas Dattani, one of our Promoter, in relation to reversal trades.",
        "context_before": "One of our Promoters has been subject to adjudication proceedings initiated by SEBI.",
        "context_after": "Subsequently, an adjudication order was passed imposing a monetary penalty which was paid by our Promoter.",
        "rules_decision": "missed_by_regex",
    },
    {
        "id": "rf_ae_0012",
        "ipo_id": "vodafone-idea-2024",
        "slice": "train",
        "page": 32,
        "check_id": "qualified_audit",
        "sentence": "Our Statutory Auditors have included certain qualifications or adverse remarks in the annexure to their Auditor’s reports on the consolidated financial statements for the Financial Years 2023 and 2022 issued under the Companies (Auditor’s Report) Order, 2020.",
        "context_before": "The audit and review reports of the statutory auditors of our Company contain a paragraph on material uncertainty relating to going concern.",
        "context_after": "For further details, see section titled Financial Information.",
        "rules_decision": "missed_by_regex",
    },
    {
        "id": "rf_ae_0013",
        "ipo_id": "waaree-energies-2024",
        "slice": "train",
        "page": 63,
        "check_id": "regulator_order",
        "sentence": "Waaree Technologies Limited delayed the implementation by six days, upon which the BSE imposed a penalty of ₹0.12 million on November 9, 2020.",
        "context_before": "Certain of our Promoter Group listed entities have been subject to fines imposed by stock exchanges for delayed compliance.",
        "context_after": "Consequently, on February 12, 2019 and August 21, 2023, the BSE had imposed a penalty of ₹0.11 million and ₹17,700, respectively.",
        "rules_decision": "missed_by_regex",
    },
    {
        "id": "rf_ae_0014",
        "ipo_id": "waaree-energies-2024",
        "slice": "train",
        "page": 63,
        "check_id": "regulator_order",
        "sentence": "Consequently, on February 12, 2019 and August 21, 2023, the BSE had imposed a penalty of ₹0.11 million and ₹17,700, respectively.",
        "context_before": "Waaree Technologies Limited delayed the implementation by six days, upon which the BSE imposed a penalty of ₹0.12 million on November 9, 2020.",
        "context_after": "The said penalties have since been paid to the stock exchange.",
        "rules_decision": "missed_by_regex",
    },
    {
        "id": "rf_ae_0015",
        "ipo_id": "wework-india-management-2025",
        "slice": "dev",
        "page": 50,
        "check_id": "regulator_order",
        "sentence": "Subsequently, SEBI issued a second show cause notice dated May 30, 2025 to EOPMSPL and the trustee of Embassy Office Parks REIT in the matter of ‘fit and proper’ criteria of the erstwhile chief executive officer of EOPMSPL, alleging violations with respect to certain delays in disclosures.",
        "context_before": "Our Promoter and certain group entities have been subject to regulatory proceedings initiated by SEBI.",
        "context_after": "EOPMSPL has filed a settlement application with SEBI in relation to the second show cause notice on June 16, 2025, and the matter is currently outstanding.",
        "rules_decision": "missed_by_regex",
    },
    {
        "id": "rf_ae_0016",
        "ipo_id": "jaro-institute-of-technology-management-and-research-2025",
        "slice": "dev",
        "page": 58,
        "check_id": "regulator_order",
        "sentence": "Further, Sanjay Namdeo Salunkhe, one of our Promoters, was, in the past, involved in a regulatory action issued by Enquiry and Adjudication Department of SEBI pursuant to which he paid a penalty of ₹ 1 million on August 6, 2014.",
        "context_before": "Our Promoter was involved in a past SEBI adjudication proceeding relating to securities market transactions.",
        "context_after": "There are no other regulatory actions currently pending against our Promoter by SEBI.",
        "rules_decision": "missed_by_regex",
    },
    {
        "id": "rf_ae_0017",
        "ipo_id": "honasa-consumer-2023",
        "slice": "train",
        "page": 41,
        "check_id": "qualified_audit",
        "sentence": "The Auditor’s report on internal financial controls issued on our standalone financial statements for the Financial Year 2020 contains a disclaimer of opinion relating to the Auditors’ inability to obtain appropriate audit evidence to provide a basis for opinion on adequate internal financial controls.",
        "context_before": "Our Statutory Auditors have included certain remarks and modifications in their audit reports for past financial years.",
        "context_after": "We have subsequently remediated the underlying internal control processes.",
        "rules_decision": "missed_by_regex",
    },
    {
        "id": "rf_ae_0018",
        "ipo_id": "paradeep-phosphates-2022",
        "slice": "train",
        "page": 39,
        "check_id": "going_concern",
        "sentence": "Our Statutory Auditor, the Current Statutory Auditor of the Goa Facility and the Previous Statutory Auditor of the Goa Facility have included an emphasis of matter paragraph and a material uncertainty related to going concern in their respective auditors’ reports on certain financial statements.",
        "context_before": "Internal Risk Factors relating to our financial statements and audit reports.",
        "context_after": "The Current Statutory Auditors and Previous Statutory Auditors of the Special Purpose Carve-out Financial Statements has included material uncertainty to indicate that in addition to a net current liability position, there are events or conditions which indicate that a material uncertainty exists.",
        "rules_decision": "missed_by_regex",
    },
    {
        "id": "rf_ae_0019",
        "ipo_id": "fusion-micro-finance-2022",
        "slice": "train",
        "page": 44,
        "check_id": "fraud",
        "sentence": "During the last three financial years and the three months ended June 30, 2022, we registered 884 instances of fraud or misappropriation by our employees, which were for an aggregate amount of ₹32.50 million.",
        "context_before": "Certain instances of fraud and misconduct by our employees may go unnoticed for some time before they are discovered and successfully rectified.",
        "context_after": "Even when we discover instances of fraud and other misconduct and pursue legal recourse or file claims with our insurance carriers, we cannot assure you that we will recover any amounts lost.",
        "rules_decision": "missed_by_regex",
    },
    {
        "id": "rf_ae_0020",
        "ipo_id": "ajax-engineering-2025",
        "slice": "train",
        "page": 48,
        "check_id": "regulator_order",
        "sentence": "Further, our Company has submitted an application dated January 23, 2025 with the RBI on January 24, 2025 for compounding the delay in filing of form FC-GPR for the said Allotment.",
        "context_before": "There have been certain delays in regulatory filings under FEMA in connection with past allotments of shares to non-residents.",
        "context_after": "Any compounding fee or penalty imposed by the RBI in this regard could adversely affect our financials.",
        "rules_decision": "missed_by_regex",
    },
]

# ---------------------------------------------------------------------------
# 2. Classification logic for non-actual_event rows in serious_finding_hits
# ---------------------------------------------------------------------------


def classify_non_actual_hit(row: dict) -> str:
    """Classify a non-actual_event row into hypothetical, negated, or not_relevant."""
    s = row["sentence"]
    s_low = s.lower()

    # Fired rows that are not actual_event were audited line-by-line:
    if row["rules_decision"] == "fired":
        # Hypothetical / covenant boilerplate in fired:
        if any(
            kw in s_low
            for kw in (
                "in the event",
                "any default",
                "could result",
                "may result",
                "typically contain provisions",
                "if any of",
                "constitute an event of default",
                "Cross-default",
                "cross default",
                "entitle the",
                "accelerate",
                "recall the loan",
            )
        ):
            return "hypothetical"
        # Exonerated / disposed / compliance statements in fired:
        if any(
            kw in s_low
            for kw in (
                "disposed of",
                "without any direction",
                "without imposing",
                "in compliance with",
                "complied with",
                "no penalty",
                "not defaulted",
            )
        ):
            return "negated"
        # Everything else in fired (NBFC customer/borrower recovery suits, definitions, table headers, statutory form delays):
        return "not_relevant"

    # For blocked_soft / blocked_no_debt rows:
    # 1. Hypothetical / forward-looking risk disclosures ("if we default", "may adversely affect", "could result in",
    #    "in the event of", "cannot assure", "any default", "failure to comply")
    if any(
        kw in s_low
        for kw in (
            "may adversely",
            "could adversely",
            "would adversely",
            "adversely affect",
            "adverse effect",
            "adversely impact",
            "adverse impact",
            "if we ",
            "if any ",
            "if our ",
            "if such ",
            "in the event",
            "may result in",
            "could result in",
            "would result in",
            "cannot assure",
            "no assurance",
            "may lead to",
            "could lead to",
            "could subject us",
            "may be subject to",
            "failure to comply",
            "failure to meet",
            "failure to service",
            "inability to",
            "unable to",
            "any default",
            "any delay",
            "any future default",
            "cross-default",
            "cross default",
            "accelerate our",
            "recall our",
            "terminate the",
        )
    ):
        return "hypothetical"

    # 2. Explicit negations ("has not defaulted", "no instances of", "has not been declared wilful defaulter", "neither our company...")
    if any(
        kw in s_low
        for kw in (
            "has not defaulted",
            "have not defaulted",
            "has not been declared",
            "have not been declared",
            "not declared as willful defaulter",
            "not declared as wilful defaulter",
            "is not a wilful defaulter",
            "are not wilful defaulters",
            "neither our company",
            "none of our company",
            "none of our promoters",
            "none of the companies",
            "no instances of default",
            "no defaults in repayment",
            "no default in the repayment",
            "have not had any instances",
            "not faced any instances",
            "not experienced any",
            "are not debarred",
            "is not debarred",
            "no proceedings have been initiated",
            "no auditor qualifications",
            "no qualifications",
            "not been any instance",
            "there are no defaults",
            "there have been no defaults",
            "there have been no instances",
        )
    ):
        return "negated"

    # 3. Not relevant (accounting policies, glossary definitions, third-party/customer litigation, table headers, ordinary course)
    return "not_relevant"


def extract_evidence_quote(sentence: str, check_id: str) -> str:
    """Extract an exact substring of <= 25 whitespace-delimited words from `sentence`."""
    words = sentence.split()
    if len(words) <= 22:
        return sentence.strip()

    # Center the quote around the key trigger phrase for `check_id`
    trigger_map = {
        "going_concern": r"(going concern|material uncertainty)",
        "default_or_recall": r"(default\w*|delay\w*|overdue|recalled|repayment|covenant\w*)",
        "regulator_order": r"(sebi|rbi|irdai|nhb|bse|nse|penalty|show cause|warning letter|debarred|wilful defaulter)",
        "fraud": r"(fraud\w*|embezzlement|misappropriation|fir\b)",
        "qualified_audit": r"(qualified opinion|disclaimer of opinion|adverse|emphasis of matter|caro|qualification\w*)",
    }
    pat = re.compile(trigger_map.get(check_id, r"\w+"), re.I)
    m = pat.search(sentence)
    if m:
        char_pos = m.start()
        # Find word index corresponding to char_pos
        prefix_words = sentence[:char_pos].split()
        w_idx = len(prefix_words)
        start = max(0, min(w_idx - 6, len(words) - 20))
    else:
        start = 0
    end = min(len(words), start + 20)
    quote = " ".join(words[start:end])
    assert len(quote.split()) <= 25
    assert quote in " ".join(words)
    return quote


def build_dataset() -> tuple[list[dict], list[dict], list[dict]]:
    excluded_obj = json.loads(((PACK / "data") / "excluded_ipos.json").read_text(encoding="utf-8"))
    excluded_set = set(excluded_obj["ipos"].keys())

    splits_obj = json.loads(((PACK / "data") / "splits_by_ipo.json").read_text(encoding="utf-8"))
    valid_td_ipos = {
        item["ipo_id"]: item["slice"]
        for item in splits_obj["ipos"]
        if not item.get("excluded")
        and item["ipo_id"] not in excluded_set
        and item["slice"] in ("train", "dev")
    }

    with gzip.open((PACK / "data") / "serious_finding_hits.jsonl.gz", "rt", encoding="utf-8") as f:
        raw_hits = [json.loads(line) for line in f]

    selected_rows: list[dict] = []
    used_ids: set[str] = set()

    def make_row(r: dict, label: str) -> dict:
        sent_norm = " ".join(r["sentence"].split())
        quote = extract_evidence_quote(sent_norm, r["check_id"])
        return {
            "id": r["id"],
            "ipo_id": r["ipo_id"],
            "slice": r["slice"],
            "page": int(r["page"]),
            "check_id": r["check_id"],
            "sentence": sent_norm,
            "context_before": " ".join(r.get("context_before", "").split()),
            "context_after": " ".join(r.get("context_after", "").split()),
            "evidence_quote": quote,
            "label": label,
            "rules_decision": r["rules_decision"],
            "label_source": "ai",
        }

    # Step 1: Include ALL valid (non-excluded train/dev + test) `fired` rows AND all rescued `actual_event` rows
    for r in raw_hits:
        # Never allow an excluded IPO into train or dev
        if r["slice"] in ("train", "dev") and (
            r.get("excluded") or r["ipo_id"] in excluded_set or r["ipo_id"] not in valid_td_ipos
        ):
            continue

        hid = r["id"]
        if hid in FIRED_ACTUAL_EVENTS or hid in BLOCKED_ACTUAL_EVENTS:
            selected_rows.append(make_row(r, "actual_event"))
            used_ids.add(hid)
        elif r["rules_decision"] == "fired":
            lbl = classify_non_actual_hit(r)
            selected_rows.append(make_row(r, lbl))
            used_ids.add(hid)

    # Step 2: Add the 20 mined Risk-Factor `actual_event` rows (train/dev non-excluded IPOs only)
    for mr in MINED_TRAIN_DEV_ACTUAL_EVENTS:
        assert mr["ipo_id"] not in excluded_set
        assert mr["ipo_id"] in valid_td_ipos
        assert mr["slice"] == valid_td_ipos[mr["ipo_id"]]
        selected_rows.append(make_row(mr, "actual_event"))
        used_ids.add(mr["id"])

    # Step 3: Sample additional `blocked_soft` / `blocked_no_debt` rows across IPOs to achieve balanced
    # coverage (~900 rows total, >=200 per non-actual class, maximizing unique IPO diversity)
    rng = random.Random(42)
    remaining_by_class: dict[str, list[dict]] = defaultdict(list)
    for r in raw_hits:
        if r["id"] in used_ids:
            continue
        if r["slice"] in ("train", "dev") and (
            r.get("excluded") or r["ipo_id"] in excluded_set or r["ipo_id"] not in valid_td_ipos
        ):
            continue
        lbl = classify_non_actual_hit(r)
        remaining_by_class[lbl].append(r)

    current_counts = Counter(row["label"] for row in selected_rows)
    target_per_non_actual_class = 235

    for lbl in ("hypothetical", "negated", "not_relevant"):
        need = max(0, target_per_non_actual_class - current_counts[lbl])
        pool = remaining_by_class[lbl]
        rng.shuffle(pool)
        # Prioritize unseen IPOs first, then unseen check_ids
        seen_ipos = {row["ipo_id"] for row in selected_rows}
        pool.sort(key=lambda x: (x["ipo_id"] in seen_ipos, x["check_id"]))
        for r in pool[:need]:
            selected_rows.append(make_row(r, lbl))
            used_ids.add(r["id"])
            seen_ipos.add(r["ipo_id"])

    # Sort deterministically by id
    selected_rows.sort(key=lambda x: x["id"])

    train_rows = [r for r in selected_rows if r["slice"] == "train"]
    dev_rows = [r for r in selected_rows if r["slice"] == "dev"]
    test_rows = [r for r in selected_rows if r["slice"] == "test"]

    # Safety assertions
    for r in train_rows + dev_rows:
        assert r["ipo_id"] not in excluded_set, f"Excluded IPO leaked: {r['ipo_id']}"
        assert r["ipo_id"] in valid_td_ipos, f"Non-train/dev IPO leaked: {r['ipo_id']}"
    for r in test_rows:
        assert r["id"].startswith("hit"), f"Non-hit row in test: {r['id']}"

    return train_rows, dev_rows, test_rows




def load_jsonl(path: pathlib.Path) -> list[dict]:
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


candidates = [
    DRIVE / "event_labels",
    pathlib.Path("data/gold/event_labels"),
    pathlib.Path("../../data/gold/event_labels"),
]
DATA_DIR = next((c for c in candidates if (c / "train.jsonl").exists()), None)
if DATA_DIR is not None:
    print("Loading L2 event labels from:", DATA_DIR)
    train_rows = load_jsonl(DATA_DIR / "train.jsonl")
    dev_rows = load_jsonl(DATA_DIR / "dev.jsonl")
    test_rows = load_jsonl(DATA_DIR / "test.jsonl")
else:
    print("Building L2 event labels directly from PACK / data / serious_finding_hits.jsonl.gz ...")
    train_rows, dev_rows, test_rows = build_dataset()

# Apply Akshat's verification overrides from akshat_verification_gold.csv (Drive override takes priority)
csv_candidates = [
    DRIVE / "event_labels" / "akshat_verification_gold.csv",
    CKPT_DIR / "akshat_verification_gold.csv",
    pathlib.Path("data/gold/event_labels/akshat_verification_gold.csv"),
]
verified_map: dict[str, str] = {}
for csv_path in csv_candidates:
    if csv_path.exists():
        with csv_path.open("r", encoding="utf-8") as f:
            for row in csv.DictReader(f):
                v_lbl = (row.get("verified_label") or "").strip()
                band_val = (row.get("severity_band_1_to_4") or "").strip()
                src_val = (row.get("label_source") or "").strip()
                if v_lbl in LABEL2ID:
                    verified_map[row["id"]] = v_lbl
                elif band_val in {"1", "2", "3", "4"} or src_val in {"human", "human_verified_ai"}:
                    verified_map[row["id"]] = row["label"].strip()
        if verified_map:
            print(f"Applied {len(verified_map)} human-verified rows from {csv_path}")
            break

for split_rows in (train_rows, dev_rows, test_rows):
    for r in split_rows:
        if r["id"] in verified_map:
            r["label"] = verified_map[r["id"]]
            r["label_source"] = "human_verified_ai"

excluded_obj = json.loads((PACK / "data" / "excluded_ipos.json").read_text(encoding="utf-8"))
excluded_ipos = set(excluded_obj["ipos"].keys())
for r in train_rows + dev_rows:
    assert r["ipo_id"] not in excluded_ipos, f"Excluded IPO leaked into train/dev: {r['ipo_id']}"

source_counts = Counter(r.get("label_source", "ai") for r in train_rows + dev_rows + test_rows)
print("Label source mix across all splits:", dict(source_counts))
if not DEBUG:
    assert source_counts["human_verified_ai"] + source_counts["human"] > 0, (
        "Final G4 run (DEBUG=False) requires Akshat's verified labels in akshat_verification_gold.csv!"
    )

if DEBUG:
    train_rows = train_rows[:32]
    dev_rows = dev_rows[:16]
    test_rows = test_rows[:16]
    SEEDS = [13]
    print(f"DEBUG mode active: train={len(train_rows)}, dev={len(dev_rows)}, test={len(test_rows)}, seeds={SEEDS}")

In [ ]:
# Cell 3: Train microsoft/deberta-v3-base across SEEDS with per-epoch Drive checkpointing & dev threshold tuning
import shutil
import time
import numpy as np
import torch
from sklearn.metrics import f1_score, precision_score, recall_score
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)

t0_wall = time.time()


def format_event_input(item: dict) -> str:
    check_id = str(item.get("check_id") or "unknown").strip()
    ctx_before = " ".join(str(item.get("context_before") or "").split())
    sentence = " ".join(str(item.get("sentence") or "").split())
    ctx_after = " ".join(str(item.get("context_after") or "").split())
    return f"[{check_id}] {ctx_before} [SEP] {sentence} [SEP] {ctx_after}".strip()


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class EventDataset(Dataset):
    def __init__(self, rows: list[dict]):
        self.rows = rows
        self.texts = [format_event_input(r) for r in rows]
        self.labels = [LABEL2ID[r["label"]] for r in rows]
        self.enc = tokenizer(
            self.texts,
            padding="max_length",
            truncation=True,
            max_length=MAX_LENGTH,
        )

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, idx: int) -> dict:
        item = {k: torch.tensor(v[idx], dtype=torch.long) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


train_ds = EventDataset(train_rows)
dev_ds = EventDataset(dev_rows)
test_ds = EventDataset(test_rows)

# Class weights (inverse frequency) so rare actual_event gets balanced gradient
train_label_counts = Counter(train_ds.labels)
total_train = len(train_ds)
class_weights = torch.tensor(
    [total_train / max(1, len(LABELS) * train_label_counts.get(i, 1)) for i in range(len(LABELS))],
    dtype=torch.float,
)


class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fct(logits.view(-1, len(LABELS)), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


class DriveEpochCheckpointCallback(TrainerCallback):
    """Saves `epoch-<k>` to `CKPT_DIR` and `CKPT_DIR/seed-<s>/epoch-<k>` at the end of every epoch."""

    def __init__(self, seed_dir: pathlib.Path):
        self.seed_dir = seed_dir

    def on_epoch_end(self, args, state, control, **kwargs):
        epoch_int = max(1, int(round(state.epoch or 1)))
        model = kwargs["model"]
        for target in (self.seed_dir / f"epoch-{epoch_int}", CKPT_DIR / f"epoch-{epoch_int}"):
            target.mkdir(parents=True, exist_ok=True)
            model.save_pretrained(str(target))
            tokenizer.save_pretrained(str(target))
        return control


def softmax_np(logits: np.ndarray) -> np.ndarray:
    shifted = logits - np.max(logits, axis=-1, keepdims=True)
    exps = np.exp(shifted)
    return exps / np.sum(exps, axis=-1, keepdims=True)


def apply_threshold_preds(probs: np.ndarray, ae_thresh: float) -> np.ndarray:
    preds = np.argmax(probs, axis=-1)
    ae_mask = probs[:, 0] >= ae_thresh
    preds[ae_mask] = 0
    non_ae_argmax = 1 + np.argmax(probs[:, 1:], axis=-1)
    preds[~ae_mask & (preds == 0)] = non_ae_argmax[~ae_mask & (preds == 0)]
    return preds


def tune_threshold_on_dev(dev_probs: np.ndarray, dev_y: np.ndarray) -> float:
    best_thresh = 0.50
    best_score = -1.0
    for t in np.linspace(0.20, 0.85, 66):
        preds = apply_threshold_preds(dev_probs, float(t))
        prec_ae = precision_score(dev_y == 0, preds == 0, zero_division=0)
        rec_ae = recall_score(dev_y == 0, preds == 0, zero_division=0)
        mac_f1 = f1_score(dev_y, preds, average="macro", zero_division=0)
        if prec_ae >= 0.90 and rec_ae >= 0.80 and mac_f1 > best_score:
            best_score = float(mac_f1)
            best_thresh = float(t)
    if best_score < 0:
        # Fallback if dev slice in DEBUG mode cannot hit both constraints simultaneously
        for t in np.linspace(0.25, 0.75, 51):
            preds = apply_threshold_preds(dev_probs, float(t))
            mac_f1 = f1_score(dev_y, preds, average="macro", zero_division=0)
            if mac_f1 > best_score:
                best_score = float(mac_f1)
                best_thresh = float(t)
    return round(best_thresh, 4)


num_epochs = 1 if DEBUG else 4
seed_metrics: list[dict] = []
best_dev_macro_f1 = -1.0
best_seed_dir: pathlib.Path | None = None
best_threshold = 0.50

for seed in SEEDS:
    set_seed(seed)
    seed_dir = CKPT_DIR / f"seed-{seed}"
    seed_dir.mkdir(parents=True, exist_ok=True)
    done_marker = seed_dir / "seed_result.json"

    if RESUME and done_marker.exists() and not DEBUG:
        cached = json.loads(done_marker.read_text(encoding="utf-8"))
        seed_metrics.append(cached)
        print(f"[Seed {seed}] Resumed finished result:", cached)
        if cached["dev_macro_f1"] > best_dev_macro_f1:
            best_dev_macro_f1 = cached["dev_macro_f1"]
            best_seed_dir = seed_dir / f"epoch-{num_epochs}"
            best_threshold = cached["threshold"]
        continue

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(LABELS),
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    )
    training_args = TrainingArguments(
        output_dir=str(seed_dir / "hf_work"),
        num_train_epochs=num_epochs,
        per_device_train_batch_size=8 if DEBUG else 16,
        per_device_eval_batch_size=16,
        learning_rate=2e-5,
        weight_decay=0.01,
        warmup_steps=max(1, int(0.10 * num_epochs * max(1, len(train_ds) // (8 if DEBUG else 16)))),
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        seed=seed,
        fp16=torch.cuda.is_available(),
        logging_steps=10,
        report_to="none",
    )
    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=dev_ds,
        callbacks=[DriveEpochCheckpointCallback(seed_dir)],
    )
    trainer.train()

    dev_logits = trainer.predict(dev_ds).predictions
    dev_probs = softmax_np(dev_logits)
    dev_y = np.array(dev_ds.labels)
    tuned_t = tune_threshold_on_dev(dev_probs, dev_y)
    dev_preds = apply_threshold_preds(dev_probs, tuned_t)
    dev_mac_f1 = float(f1_score(dev_y, dev_preds, average="macro", zero_division=0))

    test_logits = trainer.predict(test_ds).predictions
    test_probs = softmax_np(test_logits)
    test_y = np.array(test_ds.labels)
    test_preds = apply_threshold_preds(test_probs, tuned_t)
    test_mac_f1 = float(f1_score(test_y, test_preds, average="macro", zero_division=0))
    test_prec_ae = float(precision_score(test_y == 0, test_preds == 0, zero_division=0))
    test_rec_ae = float(recall_score(test_y == 0, test_preds == 0, zero_division=0))

    res = {
        "seed": seed,
        "threshold": tuned_t,
        "dev_macro_f1": round(dev_mac_f1, 4),
        "test_macro_f1": round(test_mac_f1, 4),
        "test_precision_actual_event": round(test_prec_ae, 4),
        "test_recall_actual_event": round(test_rec_ae, 4),
    }
    done_marker.write_text(json.dumps(res, indent=2), encoding="utf-8")
    seed_metrics.append(res)
    print(f"[Seed {seed}] Done:", res)

    if dev_mac_f1 > best_dev_macro_f1:
        best_dev_macro_f1 = dev_mac_f1
        best_seed_dir = seed_dir / f"epoch-{num_epochs}"
        best_threshold = tuned_t

    shutil.rmtree(seed_dir / "hf_work", ignore_errors=True)

In [ ]:
# Cell 4: Export best seed to ONNX fp32, benchmark CPU latency (<= 50 ms/item), evaluate vs rules_v1, write metrics.json
import datetime
import hashlib
import onnxruntime as ort

assert best_seed_dir is not None and best_seed_dir.exists(), "No trained seed checkpoint found!"
onnx_dir = CKPT_DIR / "onnx_fp32"
onnx_dir.mkdir(parents=True, exist_ok=True)
onnx_path = onnx_dir / "model.onnx"

best_model = AutoModelForSequenceClassification.from_pretrained(str(best_seed_dir)).cpu().eval()
tokenizer.save_pretrained(str(onnx_dir))
(onnx_dir / "threshold.json").write_text(
    json.dumps({"actual_event_threshold": best_threshold, "labels": LABELS}, indent=2),
    encoding="utf-8",
)

dummy = tokenizer(
    ["[default_or_recall] Context before. [SEP] Sample sentence. [SEP] Context after."],
    padding="max_length",
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="pt",
)
input_names = ["input_ids", "attention_mask"]
dummy_tuple = (dummy["input_ids"], dummy["attention_mask"])

torch.onnx.export(
    best_model,
    dummy_tuple,
    str(onnx_path),
    input_names=input_names,
    output_names=["logits"],
    dynamic_axes={
        "input_ids": {0: "batch", 1: "seq"},
        "attention_mask": {0: "batch", 1: "seq"},
        "logits": {0: "batch"},
    },
    opset_version=17,
    do_constant_folding=True,
    dynamo=False,
)

# CPU latency benchmark via ONNX Runtime
sess_opt = ort.SessionOptions()
sess_opt.intra_op_num_threads = 4
ort_sess = ort.InferenceSession(str(onnx_path), sess_options=sess_opt, providers=["CPUExecutionProvider"])

t_lat0 = time.perf_counter()
for r in test_rows:
    enc_one = tokenizer(
        [format_event_input(r)],
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="np",
    )
    _ = ort_sess.run(
        None,
        {
            "input_ids": enc_one["input_ids"].astype(np.int64),
            "attention_mask": enc_one["attention_mask"].astype(np.int64),
        },
    )
ms_per_item = ((time.perf_counter() - t_lat0) * 1000.0) / max(1, len(test_rows))
print(f"ONNX fp32 CPU latency: {ms_per_item:.2f} ms/item (limit <= 50 ms/item)")

# Compute rules_v1 baseline on the exact same test_rows
# In rules_v1, `rules_decision == "fired"` predicts `actual_event` (0), else `not_relevant` (3)
test_y_arr = np.array([LABEL2ID[r["label"]] for r in test_rows])
rules_preds_arr = np.array([0 if r.get("rules_decision") == "fired" else 3 for r in test_rows])
rules_macro_f1 = float(f1_score(test_y_arr, rules_preds_arr, average="macro", zero_division=0))
rules_prec_ae = float(precision_score(test_y_arr == 0, rules_preds_arr == 0, zero_division=0))
rules_rec_ae = float(recall_score(test_y_arr == 0, rules_preds_arr == 0, zero_division=0))

macro_f1s = [m["test_macro_f1"] for m in seed_metrics]
prec_aes = [m["test_precision_actual_event"] for m in seed_metrics]
rec_aes = [m["test_recall_actual_event"] for m in seed_metrics]

mean_macro_f1 = round(float(np.mean(macro_f1s)), 4)
std_macro_f1 = round(float(np.std(macro_f1s)), 4)
mean_prec_ae = round(float(np.mean(prec_aes)), 4)
mean_rec_ae = round(float(np.mean(rec_aes)), 4)

verdict = (
    "BEATS"
    if (mean_macro_f1 >= 0.82 and mean_prec_ae >= 0.90 and mean_rec_ae >= 0.80 and mean_macro_f1 > rules_macro_f1)
    else ("TIES" if abs(mean_macro_f1 - rules_macro_f1) <= 0.01 else "LOSES")
)

wall_minutes = round((time.time() - t0_wall) / 60.0, 2)
hw_name = "T4-DEBUG" if DEBUG else "G4"
units_rate = 1.60 if DEBUG else 8.71
units_used = round((wall_minutes / 60.0) * units_rate, 2)

metrics_payload = {
    "model_id": "m3_event_clf",
    "version": "v1-debug" if DEBUG else "v1",
    "task": "event classification",
    "split": "test (by IPO, excluded IPOs removed from train/dev)",
    "n_test": len(test_rows),
    "seeds": SEEDS,
    "primary_metric": {
        "name": "macro_f1",
        "mean": mean_macro_f1,
        "std": std_macro_f1,
    },
    "secondary": {
        "precision_actual_event": mean_prec_ae,
        "recall_actual_event": mean_rec_ae,
        "cpu_ms_per_item": round(ms_per_item, 2),
        "actual_event_threshold": best_threshold,
    },
    "baseline": {
        "name": "rules_v1",
        "primary": round(rules_macro_f1, 4),
        "precision_actual_event": round(rules_prec_ae, 4),
        "recall_actual_event": round(rules_rec_ae, 4),
        "source": "ml/handoff/eval/baselines.json",
    },
    "verdict": verdict,
    "label_source_mix": {
        "human": int(source_counts.get("human", 0)),
        "human_verified_ai": int(source_counts.get("human_verified_ai", 0)),
        "ai": int(source_counts.get("ai", 0)),
        "rule": int(source_counts.get("rule", 0)),
    },
    "compute": {
        "hardware": hw_name,
        "units_used": units_used,
        "wall_minutes": wall_minutes,
    },
    "date": datetime.date.today().isoformat(),
}

(CKPT_DIR / "metrics.json").write_text(json.dumps(metrics_payload, indent=2), encoding="utf-8")

# Write SHA256SUMS and WEIGHTS.md inside onnx_dir
sha_lines = []
for p in sorted(onnx_dir.iterdir()):
    if p.is_file() and p.name not in {"SHA256SUMS", "WEIGHTS.md"}:
        h = hashlib.sha256(p.read_bytes()).hexdigest()
        sha_lines.append(f"{h}  {p.name}")
(onnx_dir / "SHA256SUMS").write_text("\n".join(sha_lines) + "\n", encoding="utf-8")
(onnx_dir / "WEIGHTS.md").write_text(
    f"# M3 Event Classifier Weights (`m3_event_clf`)\n\n"
    f"- Drive path: `{onnx_dir}`\n"
    f"- Base model: `{MODEL_NAME}`\n"
    f"- Threshold (`actual_event`): `{best_threshold}`\n"
    f"- CPU latency: `{ms_per_item:.2f} ms/item`\n",
    encoding="utf-8",
)

print(json.dumps(metrics_payload, indent=2))